In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score


In [3]:
file_path = "/content/dataset 3.csv"
df = pd.read_csv(file_path)

In [4]:
# Encode target variable
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

# Define features and target
X = df.drop(columns=['label'])
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
# Binarize target labels for multi-class AUC-ROC
n_classes = len(label_encoder.classes_)
y_train_bin = label_binarize(y_train, classes=range(n_classes))
y_test_bin = label_binarize(y_test, classes=range(n_classes))

In [7]:
# Initialize models
models = {
    "SVM": OneVsRestClassifier(SVC(probability=True, random_state=42)),
    "Decision Tree": OneVsRestClassifier(DecisionTreeClassifier(random_state=42)),
    "Logistic Regression": OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
}

In [8]:
# Train models and compute AUC-ROC
auc_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train_bin)
    y_probs = model.predict_proba(X_test)
    auc = roc_auc_score(y_test_bin, y_probs, average="macro")
    auc_scores[name] = auc

In [9]:
# Print AUC scores
print("AUC-ROC Scores:")
for model, score in auc_scores.items():
    print(f"{model}: {score:.4f}")

AUC-ROC Scores:
SVM: 0.9998
Decision Tree: 0.9791
Logistic Regression: 0.9895
Random Forest: 0.9999


In [13]:
# Initialize models for accuracy calculation
from sklearn.metrics import accuracy_score # Import accuracy_score
models_accuracy = {
    "SVM": OneVsRestClassifier(SVC(probability=True, random_state=42)),
    "Decision Tree": OneVsRestClassifier(DecisionTreeClassifier(random_state=42)),
    "Logistic Regression": OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
}

# Train models and compute accuracy
accuracy_scores = {}
for name, model in models_accuracy.items():
    model.fit(X_train, y_train)  # Train on original labels
    y_pred = model.predict(X_test)  # Predict on test set
    accuracy_scores[name] = accuracy_score(y_test, y_pred)  # Compute accuracy

accuracy_scores

{'SVM': 0.9818181818181818,
 'Decision Tree': 0.95,
 'Logistic Regression': 0.9636363636363636,
 'Random Forest': 0.9931818181818182}

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train Linear Regression model
linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train)

# Make predictions
y_pred = linear_reg.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print results
print("Linear Regression Performance:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")


Linear Regression Performance:
Mean Squared Error (MSE): 28.5362
R² Score: 0.2910
